[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/04_genai_semconv/04_proyecto_genai_semconv.ipynb)

# Proyecto 04 · Modela cada paso del agente con las GenAI semantic conventions y valídalo con tu linter

| | |
|---|---|
| **Nivel** | 🟡 Intermedio |
| **Duración estimada** | ≈ 1 h 45 min |
| **Servicios de Compose** | Ninguno (todo en memoria; funciona en Colab) |
| **RAM aproximada** | < 0,5 GB |
| **Prerrequisitos** | Lección `04_genai_semconv` (spec, niveles de requisito, tokens, contenido) y módulo 03 (`OTelInstrumentor`) |
| **Conexión** | Parte de **SupportOps v1** (módulo 03). Entrega un **linter reutilizable**, un **migrador** de nombres antiguos, la **normalización de uso de tokens** y la **política de contenido** que usarán los módulos 05 (Collector), 09 (métricas) y 20 (privacidad) |

## Contexto de negocio

Eres ingeniero/a de plataforma en **SupportOps**. Un segundo equipo (el de «Facturación IA») instrumentó su agente **a la antigua usanza**: `gen_ai.system`, `prompt_tokens`, motivos de parada nativos, ids en los nombres de span, contenido de los mensajes en los atributos…
Los dashboards compartidos de coste y calidad **no funcionan con sus datos**. Tu trabajo: (1) **un linter** que mida cuánto se desvían de la spec oficial, (2) **un migrador** que los corrija en el camino, (3) la **normalización de uso y motivos de parada** para varios proveedores y (4) una **política de contenido** que no deje PII en los spans.

## Datos

Spans generados por SupportOps con **semilla fija 42** (243 spans de 60 tickets): la versión *limpia* (`OTelInstrumentor`) y la versión *legada* (`make_legacy`, con **440 defectos plantados** en 195 spans). La spec es la instantánea oficial del módulo (`genai_semconv_snapshot.json`). Cadena de resiliencia: no necesita servicios, así que no hay modo degradado.

## Entregables y criterios de evaluación (rúbrica)

| # | Entregable | Criterio medido | Umbral (calibrado con la solución, `mock`, semilla 42) |
|---|---|---|---|
| R1 | Linter · reglas de **error** (`REQ`, `UNK`, `TYPE`, `TOK`, `ERR`) | Detecta los defectos plantados de esas reglas | 281/281 (100 % de *recall*) |
| R2 | Linter · reglas de **aviso** (`NAME`, `KIND`, `FINISH`, `CARD`, `CONV`) | Detecta los defectos plantados | 138/138 (100 %) |
| R3 | Linter · `PRIV` | Marca exactamente los spans con contenido capturado | 21 spans, sin falsos positivos |
| R4 | Linter · precisión | Sobre los 243 spans **limpios**: ningún error ni aviso | 0 spans marcados |
| R5 | `migrate` | Tras migrar los spans legados, **0 errores y 0 avisos** | 243/243 spans limpios |
| R6 | `migrate` es idempotente | Migrar un span ya correcto no lo cambia | 243/243 sin cambios |
| R7 | `usage_to_semconv` | 4 formas de uso (OpenAI, Anthropic con caché, Ollama, mock) → atributos exactos | 4/4 |
| R8 | `normalize_finish` | 12 motivos de parada (+ `None` → `error`) | 13/13 |
| R9 | `redact_pii` | Sin correos, teléfonos ni tarjetas en el contenido capturado de 60 tickets con PII; el JSON sigue siendo válido | 0 PII, 100 % JSON válido |
| R10 | `to_vault` | Spans con referencia: sin correos en claro, tamaño ≤ 62 % del de captura en atributos, y toda referencia existe en el almacén | cumple (solución: 52%, de 242,118 a 126,065 bytes; 45 spans con correo → 0) |

> 📏 Los umbrales salen de lo que mide la solución de referencia con tiempo simulado y semilla fija; no dependen de tu máquina. Los defectos plantados son **deterministas** (dependen del `span_id` simulado).
> **Honestidad sobre la validez:** las comprobaciones de R1–R6 miden *recall* sobre defectos que **nosotros** plantamos; un linter con 100 % aquí puede seguir dejando pasar defectos de otro tipo. Por eso R4 y la comparación con la implementación oficial (lección, sección 4) son parte del método.

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz,otel]"], check=True)
print("Curso en:", ROOT)

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, ...
from supportops import viz, sample_data, schema, backends
viz.apply_style()
pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 80)
SEED = 42
FAST_DEV_RUN = True   # True: pocos tickets (iteración rápida). False: tamaños «completos» del texto

import copy, json, re, time, logging, collections, hashlib
from opentelemetry.sdk.trace import TracerProvider
from supportops.telemetry import setup_telemetry, OTelInstrumentor

STACK = check_stack([])                                         # este proyecto no necesita servicios
logging.getLogger("opentelemetry").setLevel(logging.ERROR)

SNAP = ROOT / "04_genai_semconv" / "genai_semconv_snapshot.json"          # instantánea de la spec oficial (ver README del módulo)
SPEC = json.loads(SNAP.read_text(encoding="utf-8"))
REG, SPANS = SPEC["attributes"], SPEC["spans"]
print("Spec:", SPEC["source"]["repo"].split("github.com/")[1], "· commit", SPEC["source"]["commit"][:8], "·", SPEC["source"]["date"], "· estado:", SPEC["source"]["status"])
print(f"{len(REG)} atributos gen_ai.* · {len(SPANS)} tipos de span · {len(SPEC['metrics'])} métricas · estabilidad de los atributos: {collections.Counter(a['stability'] for a in REG.values())}")

In [ ]:
CONTENT_ATTRS = {"gen_ai.system_instructions", "gen_ai.input.messages", "gen_ai.output.messages", "gen_ai.tool.definitions", "gen_ai.tool.call.arguments",
                 "gen_ai.tool.call.result", "gen_ai.retrieval.documents", "gen_ai.retrieval.query.text", "gen_ai.memory.records", "gen_ai.memory.query.text"}
RENAMES = {"gen_ai.system": "gen_ai.provider.name", "gen_ai.usage.prompt_tokens": "gen_ai.usage.input_tokens", "gen_ai.usage.completion_tokens": "gen_ai.usage.output_tokens",
           "gen_ai.usage.cache_creation.input_tokens": "gen_ai.usage.cache_write.input_tokens", "gen_ai.prompt": "gen_ai.input.messages (aprox.)", "gen_ai.completion": "gen_ai.output.messages (aprox.)"}
SEVERITY = {"REQ": "error", "UNK": "error", "TYPE": "error", "TOK": "error", "ERR": "error",
            "SCOPE": "warn", "ENUM": "warn", "NAME": "warn", "KIND": "warn", "FINISH": "warn", "CARD": "warn", "CONV": "warn", "PRIV": "info"}
TYPES = {"string": str, "enum": str, "int": int, "double": (int, float), "boolean": bool}
OP_TO_SPAN = {"chat": "gen_ai.client.inference", "generate_content": "gen_ai.client.inference", "text_completion": "gen_ai.client.inference", "embeddings": "gen_ai.embeddings.client",
              "retrieval": "gen_ai.retrieval.client", "create_agent": "gen_ai.create_agent.client", "execute_tool": "gen_ai.execute_tool.internal",
              "invoke_workflow": "gen_ai.invoke_workflow.internal", "plan": "gen_ai.plan.internal"}
NAME_PART = {"chat": "gen_ai.request.model", "generate_content": "gen_ai.request.model", "text_completion": "gen_ai.request.model", "embeddings": "gen_ai.request.model",
             "execute_tool": "gen_ai.tool.name", "invoke_agent": "gen_ai.agent.name", "create_agent": "gen_ai.agent.name", "plan": "gen_ai.agent.name", "retrieval": "gen_ai.data_source.id"}
HIGH_CARD = re.compile(r"(ORD-\d+|\bT-\d{3,}|\bC-\d{3,}|[0-9a-f]{8}-[0-9a-f]{4}-|\b\d{5,}\b)")      # ids que no deben ir en un nombre de span

def span_type(rec, local_agent=True):
    """Tipo de span de la spec (clave de SPANS) a partir de gen_ai.operation.name (o, si falta, de la primera palabra del nombre)."""
    op = rec["attributes"].get("gen_ai.operation.name") or rec["name"].split(" ")[0]
    if op == "invoke_agent":
        return "gen_ai.invoke_agent.internal" if (local_agent or rec["kind"] != "CLIENT") else "gen_ai.invoke_agent.client"
    return OP_TO_SPAN.get(op)

def type_ok(v, t):
    if t in (None, "any") or t.startswith("template"): return True
    if t.endswith("[]"): return isinstance(v, (list, tuple)) and all(isinstance(x, str) for x in v)
    if t in ("int", "double") and isinstance(v, bool): return False
    return isinstance(v, TYPES[t])

def finding(rule, attr, msg): return {"rule": rule, "severity": SEVERITY[rule], "attr": attr, "msg": msg}

## Los datos del proyecto

`CLEAN` son spans conformes (los de `OTelInstrumentor`); `LEGACY` los deja «a la antigua usanza» y `PLANTED[span_id]` guarda **qué reglas debería detectar tu linter** en cada uno (la verdad de terreno de la rúbrica).

In [ ]:
def _h(rec, mod): return int(rec["span_id"][:8], 16) % mod            # «azar» determinista por span

def make_legacy(rec):
    """Simula la instrumentación «de la antigua usanza» de otro equipo. Devuelve (span_modificado, defectos_plantados: set de códigos de regla)."""
    r = copy.deepcopy(rec); a = r["attributes"]; planted = set(); op = a.get("gen_ai.operation.name")
    if op == "chat":
        if "gen_ai.provider.name" in a:
            a["gen_ai.system"] = a.pop("gen_ai.provider.name"); planted |= {"UNK", "REQ"}                 # nombre antiguo del proveedor
        if "gen_ai.usage.input_tokens" in a:
            a["gen_ai.usage.prompt_tokens"] = a.pop("gen_ai.usage.input_tokens"); a["gen_ai.usage.completion_tokens"] = a.pop("gen_ai.usage.output_tokens"); planted.add("UNK")
        if a.get("gen_ai.response.finish_reasons") == ["tool_call"]:
            a["gen_ai.response.finish_reasons"] = ["tool_calls"]; planted.add("FINISH")                  # valor «nativo» sin normalizar
        if _h(r, 6) == 0:
            a["gen_ai.input.messages"] = json.dumps([{"role": "user", "parts": [{"type": "text", "content": "Hola, soy un cliente"}]}]); planted.add("PRIV")
    elif op == "execute_tool":
        if _h(r, 3) == 0:
            a["tool"] = a.pop("gen_ai.tool.name"); planted.add("REQ")                                    # atributo propio en lugar del estándar
        elif _h(r, 5) == 0:
            r["name"] += " ORD-00017"; planted |= {"CARD", "NAME"}                                       # id dentro del nombre del span
    elif op == "invoke_agent":
        if _h(r, 2) == 0:
            r["kind"] = "CLIENT"; planted.add("KIND")                                                    # agente local marcado como remoto
        if _h(r, 4) == 0 and "gen_ai.usage.input_tokens" in a:
            a["gen_ai.usage.input_tokens"] = str(a["gen_ai.usage.input_tokens"]); planted.add("TYPE")    # número como cadena
        elif _h(r, 7) == 0 and "gen_ai.usage.input_tokens" in a:
            a["gen_ai.usage.cache_read.input_tokens"] = a["gen_ai.usage.input_tokens"] + 50; planted.add("TOK")   # subconjunto mayor que el total
        if _h(r, 3) == 0:
            a["gen_ai.conversation.id"] = r["trace_id"]; planted.add("CONV")                             # conversación inventada con el trace_id
    if r["status"] == "ERROR" and _h(r, 2) == 0 and "error.type" in a:
        a.pop("error.type"); planted.add("ERR")                                                          # error sin clasificar
    return r, planted

In [ ]:
tickets = generate_tickets(60, seed=SEED, fault_rates=DEFAULT_FAULT_RATES, start_s=SIM_EPOCH_S)
tel = setup_telemetry("supportops-agent", mode="memory", set_global=False, deterministic_ids=True)      # ids deterministas: los defectos plantados son siempre los mismos
run_tickets(SupportAgent(instrumentor=OTelInstrumentor(tel)), tickets)
CLEAN = tel.span_records()
_pairs = [make_legacy(s) for s in CLEAN]
LEGACY, PLANTED = [p[0] for p in _pairs], {s["span_id"]: p[1] for s, p in zip(CLEAN, _pairs)}
print(len(CLEAN), "spans limpios ·", sum(bool(p) for p in PLANTED.values()), "legados con algún defecto ·", sum(len(p) for p in PLANTED.values()), "defectos plantados")
print("ejemplo de span legado:", {k: v for k, v in LEGACY[0]["attributes"].items() if k.startswith("gen_ai.")})

## Paso 1 · El linter, por familias de reglas

Tienes disponibles `SPEC`, `REG` (atributos), `SPANS` (tipos de span), `span_type(rec)`, `type_ok(valor, tipo)`, `RENAMES`, `NAME_PART`, `HIGH_CARD`, `CONTENT_ATTRS`, `SEVERITY` y `finding(regla, atributo, mensaje)`. Un *span* `rec` es un `dict` con `name`, `kind`, `status`, `trace_id`, `attributes`…
Cada función devuelve una **lista de hallazgos** (`finding(...)`). La tabla de reglas está en la lección (sección 4).

**TODO 1 · `check_attributes(rec)`:** `UNK` (atributo `gen_ai.*` que no existe en `REG`, sugiriendo `RENAMES[k]` si lo conoces), `TYPE` (valor que no cumple `type_ok`) y `SCOPE` (atributo del registro que no está definido para el tipo de span).

<details>
<summary>🪜 Pista 1</summary>

Recorre `rec['attributes'].items()`. Un atributo es **desconocido** si empieza por `gen_ai.`, no está en `REG` y no empieza por `gen_ai.prompt.variable.` (es una plantilla). Si está en `REG`, comprueba el tipo con `type_ok(v, REG[k]['type'])`.

</details>

<details>
<summary>🪜 Pista 2</summary>

Para `SCOPE`: `st = span_type(rec)`; `defined = SPANS[st]['attributes'] if st else {}`; avisa si `k in REG` pero `k not in defined`.

</details>

In [ ]:
def check_attributes(rec):
    # TODO 1
    raise NotImplementedError("TODO 1")

**TODO 2 · `check_required(rec)`:** `REQ` por cada atributo con `level == "required"` en `SPANS[tipo]["attributes"]` que falte, y `ERR` si `rec["status"] == "ERROR"` y no hay `error.type`.

<details>
<summary>🪜 Pista 1</summary>

`SPANS[st]['attributes']` es un dict `{atributo: {'level': 'required'|'conditionally_required'|'recommended'|'opt_in', ...}}`. Si `span_type(rec)` es `None` (operación desconocida) no hay requeridos que comprobar.

</details>

In [ ]:
def check_required(rec):
    # TODO 2
    raise NotImplementedError("TODO 2")

**TODO 3 · `check_shape(rec)`:** `NAME` (el nombre debe ser `{operación} {valor del atributo de NAME_PART[operación]}`; para `invoke_agent` sin nombre de agente basta `invoke_agent`; si falta el atributo del que depende el nombre, **no** avises), `KIND` (el *kind* del span debe ser `SPANS[tipo]["kind"].upper()`; para `gen_ai.client.inference` se admite también `INTERNAL`) y `CARD` (`HIGH_CARD.search(nombre)`).

<details>
<summary>🪜 Pista 1</summary>

`op = a.get('gen_ai.operation.name')`; `part = a.get(NAME_PART[op])`; `expected = op + (' ' + part if part else '')`.

</details>

In [ ]:
def check_shape(rec):
    # TODO 3
    raise NotImplementedError("TODO 3")

**TODO 4 · `check_values(rec)`:** `FINISH` (cada valor de `gen_ai.response.finish_reasons` debe estar en `SPEC["finish_reason_enum"]`), `TOK` (`cache_read`/`cache_write` no pueden superar `gen_ai.usage.input_tokens`), `CONV` (`gen_ai.conversation.id` igual a `rec["trace_id"]`) y `PRIV` (un hallazgo por cada atributo de `CONTENT_ATTRS` presente).

<details>
<summary>🪜 Pista 1</summary>

Comprueba antes de comparar que los valores son `int` (un `str` ya habrá dado `TYPE` en el TODO 1 y compararlo lanzaría una excepción). `CONTENT_ATTRS & set(a)` da los atributos de contenido presentes.

</details>

In [ ]:
def check_values(rec):
    # TODO 4
    raise NotImplementedError("TODO 4")

def lint_span(rec):
    return check_attributes(rec) + check_required(rec) + check_shape(rec) + check_values(rec)

try:
    r0 = next(s for s in LEGACY if s["name"].startswith("chat"))
    print([ (f["rule"], f["attr"]) for f in lint_span(r0)])
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 2 · El migrador

**TODO 5.** Implementa `migrate(rec) -> rec` (devuelve una **copia**; no mutes la entrada) que deje un span legado **sin errores ni avisos** del linter y que **no cambie** un span ya correcto (idempotente):

1. aplica los **renombres exactos** de `RENAMES` (los que no son «aprox.»);
2. `tool` → `gen_ai.tool.name`;
3. números que llegaron como cadena → `int` (solo atributos que el registro declara `int`);
4. `finish_reasons` al enum de la spec (usa `normalize_finish`, TODO 6b);
5. **elimina** los atributos de contenido (una migración no debe arrastrar PII);
6. quita `cache_read`/`cache_write` incoherentes (mayores que la entrada);
7. quita `gen_ai.conversation.id` si es el `trace_id`;
8. un span `ERROR` sin tipo recibe `error.type = "_OTHER"` (el valor de reserva de la spec);
9. `invoke_agent` pasa a `INTERNAL` (nuestro agente es local);
10. quita identificadores del nombre del span (`HIGH_CARD.sub("", nombre).strip()`).

<details>
<summary>🪜 Pista 1</summary>

Empieza por `r = copy.deepcopy(rec)` y modifica `r['attributes']` en sitio. Itera sobre `list(a)` si vas a borrar claves mientras recorres.

</details>

<details>
<summary>🪜 Pista 2</summary>

Puedes escribir primero el TODO 6b (`normalize_finish`) y llamarlo desde aquí. Para la idempotencia, cada paso debe ser un no-op cuando el span ya cumple (p. ej. `if old in a`).

</details>

In [ ]:
def migrate(rec):
    # TODO 5
    raise NotImplementedError("TODO 5")

## Paso 3 · Normalizar uso de tokens y motivos de parada

**TODO 6a · `usage_to_semconv(provider, usage) -> dict`.** Dado el objeto de uso **tal como lo devuelve cada proveedor**, devuelve los atributos `gen_ai.usage.*` de la spec (solo los que el proveedor informa):

| Proveedor | Forma del uso | Notas |
|---|---|---|
| `openai` | `prompt_tokens`, `completion_tokens`, `prompt_tokens_details.cached_tokens`, `completion_tokens_details.reasoning_tokens` | `prompt_tokens` **ya incluye** lo cacheado |
| `anthropic` | `input_tokens`, `output_tokens`, `cache_read_input_tokens`, `cache_creation_input_tokens` | `input_tokens` **excluye** caché → suma |
| `ollama` | `prompt_eval_count`, `eval_count` | |
| `mock` (SupportOps) | `input_tokens`, `output_tokens` | |

**TODO 6b · `normalize_finish(provider, reason)`.** Traduce el motivo nativo al enum de la spec (`stop`, `length`, `tool_call`, `content_filter`, `error`): OpenAI/mock `stop`, `length`, `tool_calls`, `function_call`, `content_filter`; Anthropic `end_turn`, `stop_sequence`, `max_tokens`, `tool_use`, `refusal`; Ollama `stop`, `length`.
`None` o vacío → `"error"` (la spec pide reportar `error` cuando se esperaba un motivo y no llegó); un valor desconocido se devuelve tal cual (así el linter lo señalará).

<details>
<summary>🪜 Pista 1</summary>

Usa `usage.get('cache_read_input_tokens', 0)` para sumar sin fallar cuando el proveedor no informa caché, pero **solo escribe el atributo `cache_read`** si el proveedor lo informó.

</details>

<details>
<summary>🪜 Pista 2</summary>

Para los motivos, un diccionario por proveedor y `.get(reason, reason)`.

</details>

In [ ]:
def usage_to_semconv(provider: str, usage: dict) -> dict:
    # TODO 6a
    raise NotImplementedError("TODO 6a")

def normalize_finish(provider: str, reason):
    # TODO 6b
    raise NotImplementedError("TODO 6b")

try:
    print(usage_to_semconv("anthropic", {"input_tokens": 50, "output_tokens": 30, "cache_read_input_tokens": 100, "cache_creation_input_tokens": 20}))
    print(normalize_finish("anthropic", "tool_use"), normalize_finish("openai", None))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 4 · Política de contenido

Capturamos contenido (`capture_content=True`) de 60 tickets con **30 % de PII sintética** para probar la política.

**TODO 7a · `redact_pii(text) -> str`:** sustituye **tarjetas** (`4000 0000 9705 5284`), **correos** y **teléfonos** (`+34 625 989 712`) por marcadores (`[TARJETA]`, `[EMAIL]`, `[TELEFONO]`). Cuidado con el **orden** (una tarjeta también «parece» un teléfono) y con los **falsos positivos**: un regex de teléfono demasiado general convierte en `[TELEFONO]` los números de seguimiento (`TRK…`) y las marcas de tiempo, y **rompe el JSON** del contenido (la rúbrica lo comprueba).
**TODO 7b · `to_vault(text, vault) -> str`:** guarda `text` en el diccionario `vault` bajo un hash corto (p. ej. `sha256(...)[:16]`) y devuelve el JSON `{"vault_ref": "<hash>"}`, que es lo que irá al span (patrón 3 de la lección; la spec aún no define cómo referenciar contenido externo).

<details>
<summary>🪜 Pista 1</summary>

Regex: correo `r"[\w.+-]+@[\w-]+\.[\w.-]+"`; tarjeta `r"\b(?:\d{4} ){3}\d{4}\b"`; teléfono internacional `r"\+\d{1,3}(?: \d{2,4}){2,4}"` (exige el `+`: un patrón más general también cazaría números de seguimiento y marcas de tiempo y **rompería el JSON**). Aplica primero tarjeta.

</details>

In [ ]:
import re, hashlib
tk_pii = generate_tickets(60, seed=SEED, fault_rates=DEFAULT_FAULT_RATES, pii_rate=0.3, start_s=SIM_EPOCH_S)

def redact_pii(text: str) -> str:
    # TODO 7a
    raise NotImplementedError("TODO 7a")

def to_vault(text: str, vault: dict) -> str:
    # TODO 7b
    raise NotImplementedError("TODO 7b")

EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+"); PHONE_RE = re.compile(r"\+\d{1,3}(?: \d{2,4}){2,4}"); CARD_RE = re.compile(r"\b(?:\d{4} ){3}\d{4}\b")   # (los usa la rúbrica para medir fugas)
def _pii_count(spans):                       # spans con PII en claro DENTRO de los atributos de contenido (los ids como resp_6ba07335… no son PII)
    txt = lambda x: " ".join(str(v) for k, v in x["attributes"].items() if k in CONTENT_ATTRS)
    return sum(bool(EMAIL_RE.search(txt(x)) or PHONE_RE.search(txt(x)) or CARD_RE.search(txt(x))) for x in spans)
try:
    print(redact_pii("Mi correo es ana@example.com y mi teléfono +34 625 989 712, tarjeta 4000 0000 9705 5284."))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 5 · Rúbrica

Ejecuta la celda para puntuar tu trabajo.

In [ ]:
def _spans_with_policy(**kw):
    t = setup_telemetry("policy", mode="memory", set_global=False)
    run_tickets(SupportAgent(instrumentor=OTelInstrumentor(t, **kw)), tk_pii); sp = t.span_records(); t.shutdown(); return sp

def run_rubric():
    rows = []
    def add(r, desc, ok, detail=""): rows.append({"criterio": r, "descripción": desc, "resultado": "✅ OK" if ok else "❌ FALLA", "detalle": detail})
    # ---- R1-R4: linter
    try:
        rules_of = lambda rec: {f["rule"] for f in lint_span(rec)}
        LF = {s["span_id"]: rules_of(s) for s in LEGACY}
        def rec_of(group):
            exp = [(sid, r) for sid, p in PLANTED.items() for r in p if r in group]; hit = sum(r in LF[sid] for sid, r in exp); return hit, len(exp)
        h, n = rec_of({"REQ", "UNK", "TYPE", "TOK", "ERR"}); add("R1", "reglas de error: defectos plantados detectados", h == n, f"{h}/{n}")
        h, n = rec_of({"NAME", "KIND", "FINISH", "CARD", "CONV"}); add("R2", "reglas de aviso: defectos plantados detectados", h == n, f"{h}/{n}")
        exp_priv = {sid for sid, p in PLANTED.items() if "PRIV" in p}; got_priv = {sid for sid, r in LF.items() if "PRIV" in r}
        add("R3", "PRIV exacto", exp_priv == got_priv, f"esperados {len(exp_priv)}, marcados {len(got_priv)}")
        bad = [s["name"] for s in CLEAN if any(f["severity"] in ("error", "warn") for f in lint_span(s))]
        add("R4", "spans limpios sin errores ni avisos", not bad, f"{len(bad)} marcados" + (f" (p. ej. {bad[0]})" if bad else ""))
    except Exception as e:
        for r in ("R1", "R2", "R3", "R4"): add(r, "linter", False, repr(e)[:80])
    # ---- R5-R6: migrador
    try:
        m = [migrate(s) for s in LEGACY]; bad = [s["name"] for s in m if any(f["severity"] in ("error", "warn") for f in lint_span(s))]
        add("R5", "tras migrar: 0 errores y 0 avisos", not bad, f"{len(m) - len(bad)}/{len(m)} limpios")
        same = sum(migrate(s) == s for s in CLEAN); add("R6", "migrate idempotente en spans correctos", same == len(CLEAN), f"{same}/{len(CLEAN)}")
    except Exception as e:
        for r in ("R5", "R6"): add(r, "migrate", False, repr(e)[:80])
    # ---- R7-R8: uso y motivos de parada
    try:
        cases = [("openai", {"prompt_tokens": 1200, "completion_tokens": 150, "prompt_tokens_details": {"cached_tokens": 900}, "completion_tokens_details": {"reasoning_tokens": 100}},
                  {"gen_ai.usage.input_tokens": 1200, "gen_ai.usage.output_tokens": 150, "gen_ai.usage.cache_read.input_tokens": 900, "gen_ai.usage.reasoning.output_tokens": 100}),
                 ("anthropic", {"input_tokens": 50, "output_tokens": 30, "cache_read_input_tokens": 100, "cache_creation_input_tokens": 20},
                  {"gen_ai.usage.input_tokens": 170, "gen_ai.usage.output_tokens": 30, "gen_ai.usage.cache_read.input_tokens": 100, "gen_ai.usage.cache_write.input_tokens": 20}),
                 ("ollama", {"prompt_eval_count": 90, "eval_count": 25}, {"gen_ai.usage.input_tokens": 90, "gen_ai.usage.output_tokens": 25}),
                 ("mock", {"input_tokens": 10, "output_tokens": 3}, {"gen_ai.usage.input_tokens": 10, "gen_ai.usage.output_tokens": 3})]
        ok = sum(usage_to_semconv(p, u) == e for p, u, e in cases); add("R7", "uso de tokens de 4 proveedores", ok == 4, f"{ok}/4")
        fin = [("openai", "stop", "stop"), ("openai", "length", "length"), ("openai", "tool_calls", "tool_call"), ("openai", "content_filter", "content_filter"), ("mock", "tool_calls", "tool_call"),
               ("anthropic", "end_turn", "stop"), ("anthropic", "max_tokens", "length"), ("anthropic", "tool_use", "tool_call"), ("anthropic", "refusal", "content_filter"),
               ("anthropic", "stop_sequence", "stop"), ("ollama", "stop", "stop"), ("ollama", "length", "length"), ("openai", None, "error")]
        ok = sum(normalize_finish(p, r) == e for p, r, e in fin); add("R8", "motivos de parada normalizados", ok == len(fin), f"{ok}/{len(fin)}")
    except Exception as e:
        for r in ("R7", "R8"): add(r, "uso/finish", False, repr(e)[:80])
    # ---- R9-R10: política de contenido
    try:
        raw = _spans_with_policy(capture_content=True)
        red = _spans_with_policy(capture_content=True, redact=redact_pii)
        def valid_json(sp): return all(json.loads(s["attributes"][k]) is not None for s in sp for k in CONTENT_ATTRS if k in s["attributes"])
        add("R9", "redact_pii: 0 PII y JSON válido", _pii_count(raw) > 0 and _pii_count(red) == 0 and valid_json(red), f"con PII en claro: {_pii_count(raw)} spans → tras redactar: {_pii_count(red)}")
        vault = {}; vlt = _spans_with_policy(capture_content=True, redact=lambda s: to_vault(s, vault))
        size = lambda sp: sum(len(json.dumps(s["attributes"], ensure_ascii=False).encode()) for s in sp)
        refs = [json.loads(s["attributes"][k])["vault_ref"] for s in vlt for k in CONTENT_ATTRS if k in s["attributes"]]
        ratio = size(vlt) / size(raw)
        add("R10", "almacén: sin correos en claro, tamaño reducido, referencias válidas", _pii_count(vlt) == 0 and ratio <= REF_RATIO_MAX and bool(refs) and all(r in vault for r in refs), f"tamaño = {ratio:.0%} del de captura en atributos")
    except Exception as e:
        for r in ("R9", "R10"): add(r, "política de contenido", False, repr(e)[:80])
    return pd.DataFrame(rows).set_index("criterio")

REF_RATIO_MAX = 0.62
display(run_rubric())

---
## Retos extra (nivel experto)

1. **Linter como `SpanProcessor` de aviso:** en lugar de analizar spans guardados, implementa un procesador que, al terminar cada span, ejecute `lint_span` y **cuente** los hallazgos con un `Counter` (y emita una métrica `supportops.semconv.findings{rule}`). ¿Qué coste añade por span? (mídelo como en el módulo 03).
2. **Linter de métricas:** extiende el linter a nombres, unidades e instrumentos de métricas usando `SPEC["metrics"]` (p. ej. detectar `gen_ai.client.token.usage` como obsoleta).
3. **Doble emisión:** modifica `OTelInstrumentor` (subclase) para emitir **también** `gen_ai.usage.prompt_tokens`/`completion_tokens` durante una transición, con una variable de entorno que lo active. Demuestra con el linter que los spans «dobles» generan `UNK` solo por los nombres antiguos.
4. **Redacción más fina:** añade IBAN y DNI sintéticos a `redact_pii`, mide falsos positivos con números de pedido (`ORD-00017`) y razona la diferencia entre *redactar* (irreversible) y *seudonimizar* (hash con sal que permite correlacionar).
5. **Contra el oráculo:** instala `opentelemetry-util-genai` y comprueba que su salida pasa tu linter para `invoke_agent`, `chat` y `execute_tool`; si alguna regla discrepa, decide quién tiene razón **leyendo la spec**.

## Reflexión (producción e hilo conductor)

- Tu linter dio 100 % de *recall* sobre defectos que **plantamos nosotros**. ¿Qué tipos de defecto **no** puede detectar un linter basado en la spec (pista: valores correctos pero falsos, `gen_ai.request.model` equivocado, tokens mal contados por el proveedor)?
- La spec está en `development`: ¿qué proceso pondrías para que una nueva versión de la spec no rompa tus dashboards (versión fijada, CI con el linter, doble emisión, fecha de retirada)?
- El patrón 3 mueve la PII al almacén. ¿Quién debe poder leerlo, durante cuánto tiempo y cómo auditarías los accesos? (módulo 20)
- En el módulo 05 pondrás un **Collector** entre los servicios y los backends: ¿qué parte de `migrate` y `redact_pii` preferirías ejecutar *en el Collector* y cuál *en el SDK*? ¿Por qué?

---
# ⛔ SPOILER — intenta resolverlo primero

A continuación, la **solución de referencia** completa y ejecutable, y la rúbrica con ella.

In [ ]:
# ===================== SOLUCIÓN DE REFERENCIA =====================
# TODO 6a · uso de tokens de cada proveedor -> atributos de la spec
def usage_to_semconv(provider: str, usage: dict) -> dict:
    out = {}
    if provider == "anthropic":                                                # input_tokens EXCLUYE caché
        rd, wr = usage.get("cache_read_input_tokens", 0), usage.get("cache_creation_input_tokens", 0)
        out["gen_ai.usage.input_tokens"] = usage["input_tokens"] + rd + wr
        out["gen_ai.usage.output_tokens"] = usage["output_tokens"]
        if "cache_read_input_tokens" in usage: out["gen_ai.usage.cache_read.input_tokens"] = rd
        if "cache_creation_input_tokens" in usage: out["gen_ai.usage.cache_write.input_tokens"] = wr
    elif provider == "openai":                                                 # prompt_tokens YA incluye lo cacheado
        out["gen_ai.usage.input_tokens"], out["gen_ai.usage.output_tokens"] = usage["prompt_tokens"], usage["completion_tokens"]
        cached = usage.get("prompt_tokens_details", {}).get("cached_tokens")
        reasoning = usage.get("completion_tokens_details", {}).get("reasoning_tokens")
        if cached is not None: out["gen_ai.usage.cache_read.input_tokens"] = cached
        if reasoning is not None: out["gen_ai.usage.reasoning.output_tokens"] = reasoning
    elif provider == "ollama":
        out["gen_ai.usage.input_tokens"], out["gen_ai.usage.output_tokens"] = usage["prompt_eval_count"], usage["eval_count"]
    else:                                                                      # mock / SupportOps
        out["gen_ai.usage.input_tokens"], out["gen_ai.usage.output_tokens"] = usage["input_tokens"], usage["output_tokens"]
    return out

# TODO 6b · motivo de parada nativo -> enum de la spec (None -> "error"; lo desconocido se deja tal cual)
FINISH_MAP = {"openai": {"stop": "stop", "length": "length", "tool_calls": "tool_call", "function_call": "tool_call", "content_filter": "content_filter"},
              "anthropic": {"end_turn": "stop", "stop_sequence": "stop", "max_tokens": "length", "tool_use": "tool_call", "refusal": "content_filter"},
              "ollama": {"stop": "stop", "length": "length"}}
FINISH_MAP["mock"] = FINISH_MAP["openai"]

def normalize_finish(provider: str, reason):
    if not reason: return "error"
    return FINISH_MAP.get(provider, {}).get(reason, reason)

In [ ]:
# TODO 1 · atributos: UNK (con sugerencia), TYPE, SCOPE
def check_attributes(rec):
    F, a, st = [], rec["attributes"], span_type(rec)
    defined = SPANS[st]["attributes"] if st else {}
    for k, v in a.items():
        if k.startswith("gen_ai.") and k not in REG and not k.startswith("gen_ai.prompt.variable."):
            F.append(finding("UNK", k, "no existe en el registro actual" + (f"; usa {RENAMES[k]}" if k in RENAMES else "")))
        elif k in REG:
            if not type_ok(v, REG[k]["type"]): F.append(finding("TYPE", k, f"esperaba {REG[k]['type']}, llegó {type(v).__name__}"))
            if st and k not in defined: F.append(finding("SCOPE", k, f"no está definido para {st}"))
    return F

# TODO 2 · requeridos: REQ y ERR
def check_required(rec):
    F, a, st = [], rec["attributes"], span_type(rec)
    if st:
        F += [finding("REQ", k, f"requerido en {st}") for k, d in SPANS[st]["attributes"].items() if d["level"] == "required" and k not in a]
    if rec["status"] == "ERROR" and "error.type" not in a:
        F.append(finding("ERR", "error.type", "span en ERROR sin error.type"))
    return F

# TODO 3 · forma del span: NAME, KIND, CARD
def check_shape(rec):
    F, a, st = [], rec["attributes"], span_type(rec)
    op = a.get("gen_ai.operation.name")
    if st and op in NAME_PART:
        part = a.get(NAME_PART[op])
        if part is not None or op == "invoke_agent":
            expected = op + (f" {part}" if part else "")
            if rec["name"] != expected: F.append(finding("NAME", "name", f"esperado «{expected}», hay «{rec['name']}»"))
    if st:
        exp_kind = SPANS[st]["kind"].upper()
        if rec["kind"] != exp_kind and not (st == "gen_ai.client.inference" and rec["kind"] == "INTERNAL"):
            F.append(finding("KIND", "kind", f"{st} debería ser {exp_kind}"))
    if HIGH_CARD.search(rec["name"]): F.append(finding("CARD", "name", "el nombre del span contiene un identificador"))
    return F

# TODO 4 · valores: FINISH, TOK, CONV, PRIV
def check_values(rec):
    F, a = [], rec["attributes"]
    fr = a.get("gen_ai.response.finish_reasons")
    for x in (fr if isinstance(fr, (list, tuple)) else []):
        if x not in SPEC["finish_reason_enum"]: F.append(finding("FINISH", "gen_ai.response.finish_reasons", f"{x!r} no está en {SPEC['finish_reason_enum']}"))
    i = a.get("gen_ai.usage.input_tokens")
    if isinstance(i, int) and not isinstance(i, bool):
        for sub in ("gen_ai.usage.cache_read.input_tokens", "gen_ai.usage.cache_write.input_tokens"):
            if isinstance(a.get(sub), int) and a[sub] > i: F.append(finding("TOK", sub, f"{sub}={a[sub]} > input_tokens={i}"))
    if a.get("gen_ai.conversation.id") == rec["trace_id"]: F.append(finding("CONV", "gen_ai.conversation.id", "es el trace_id"))
    F += [finding("PRIV", k, f"contenido capturado ({len(str(a[k]))} car.)") for k in sorted(CONTENT_ATTRS & set(a))]
    return F

def lint_span(rec):
    return check_attributes(rec) + check_required(rec) + check_shape(rec) + check_values(rec)

In [ ]:
# TODO 5 · migrar un span «legado» a la spec actual (devuelve una copia)
def migrate(rec):
    r = copy.deepcopy(rec); a = r["attributes"]
    for old, new in RENAMES.items():                                           # 1) renombres exactos
        if old in a and "(aprox.)" not in new: a[new] = a.pop(old)
    if "tool" in a and "gen_ai.tool.name" not in a: a["gen_ai.tool.name"] = a.pop("tool")      # 2) atributo propio -> estándar
    for k in list(a):                                                          # 3) tipos: números que llegaron como cadena
        if k in REG and REG[k]["type"] == "int" and isinstance(a[k], str) and a[k].isdigit(): a[k] = int(a[k])
    fr = a.get("gen_ai.response.finish_reasons")                               # 4) finish reasons al enum de la spec
    if isinstance(fr, list): a["gen_ai.response.finish_reasons"] = [normalize_finish("mock", x) for x in fr]
    for k in CONTENT_ATTRS & set(a): a.pop(k)                                  # 5) privacidad: la migración no arrastra contenido
    i = a.get("gen_ai.usage.input_tokens")                                     # 6) subconjuntos de tokens coherentes
    for sub in ("gen_ai.usage.cache_read.input_tokens", "gen_ai.usage.cache_write.input_tokens"):
        if isinstance(i, int) and isinstance(a.get(sub), int) and a[sub] > i: a.pop(sub)
    if a.get("gen_ai.conversation.id") == r["trace_id"]: a.pop("gen_ai.conversation.id")      # 7) conversación inventada
    if r["status"] == "ERROR" and "error.type" not in a: a["error.type"] = "_OTHER"          # 8) error sin clasificar
    op = a.get("gen_ai.operation.name")
    if op == "invoke_agent": r["kind"] = "INTERNAL"                            # 9) agente en proceso
    r["name"] = HIGH_CARD.sub("", r["name"]).strip()                           # 10) ids fuera del nombre
    return r

In [ ]:
# TODO 7 · política de contenido: redacción y almacén con referencia
EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
CARD_RE = re.compile(r"\b(?:\d{4} ){3}\d{4}\b")
PHONE_RE = re.compile(r"\+\d{1,3}(?: \d{2,4}){2,4}")

def redact_pii(text: str) -> str:
    text = CARD_RE.sub("[TARJETA]", text)                                      # primero tarjetas (también son dígitos con espacios)
    text = EMAIL_RE.sub("[EMAIL]", text)
    return PHONE_RE.sub("[TELEFONO]", text)

def to_vault(text: str, vault: dict) -> str:
    h = hashlib.sha256(text.encode()).hexdigest()[:16]; vault[h] = text       # el contenido (con su política de acceso) vive FUERA del span
    return json.dumps({"vault_ref": h})

In [ ]:
# Re-ejecuta la rúbrica con la solución
display(run_rubric())